# Sprint 1 Baseline RAG Evaluation — Fresh Colab Run

## Objective

Run the cleaned Sprint 1 RAG project from scratch in Google Colab.

We evaluate four retrievers:

1. BM25
2. Original DPR
3. Contriever
4. ColBERTv2

The generator is fixed for all retrievers:

- Provider: Mannheim Maki API
- Model: ministral-3-14b
- Temperature: 0.0
- Max tokens: 512
- Top-k: 5

This matches the Sprint 1 baseline requirement: a working end-to-end RAG system with a fixed generator and interchangeable retrievers.

In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
from google.colab import files

uploaded = files.upload()

Saving medical-rag-maki-colab.zip to medical-rag-maki-colab.zip


In [20]:
import os
from pathlib import Path

# Reset current working directory safely
os.chdir("/content")

SEARCH_ROOT = Path("/content/medical-rag-maki-colab")

matches = []

for path in SEARCH_ROOT.rglob("eval_all_retrievers_safe.py"):
    candidate = path.parent
    if (candidate / "retrievers").exists() and (candidate / "requirements.txt").exists():
        matches.append(candidate)

if not matches:
    print("No valid project root found.")
    print("Showing files:")
    !find /content/medical-rag-maki-colab -maxdepth 5 -type f | head -100
    raise RuntimeError("Project root not found.")

project_root = matches[0]
os.chdir(project_root)

print("✅ Project root found:")
print(os.getcwd())

print("\nRoot files:")
!ls

print("\nRetriever files:")
!ls retrievers

✅ Project root found:
/content/medical-rag-maki-colab/medical-rag-maki-colab

Root files:
 app.py				        pipeline.py
 archive			        pubmed_fetcher.py
 colbertv2_antigravity_colab_guide.md   pubmed_fetch.py
 config.py			        __pycache__
 data				        README.md
 data_prep.py			        requirements.txt
 eval_all_retrievers.py		        results
 eval_all_retrievers_safe.py	        retrievers
 evaluation			        run_eval_colbert.bat
 generator.py			        SPRINT1_REPRODUCTION.md
 make_colab_zip.py		       'Team Project Kick-Off (4).pdf'
 notebooks

Retriever files:
bm25_retriever.py     dense_retriever.py	 factory.py   live_retriever.py
colbert_retriever.py  dpr_original_retriever.py  __init__.py  __pycache__


# Install Dependencies

This cell installs dependencies in the correct order.

Important previous errors fixed here:

1. NumPy/Pandas binary mismatch
2. RAGatouille missing `langchain.retrievers`
3. ColBERTv2 compatibility issue

In [21]:
# Base project dependencies
!pip -q install \
  faiss-cpu \
  rank-bm25 \
  datasets \
  transformers \
  accelerate \
  sentence-transformers \
  tqdm \
  scikit-learn \
  rouge-score \
  requests

# RAGatouille for ColBERTv2
!pip -q install ragatouille

# Required old LangChain modules for RAGatouille compatibility
!pip -q install \
  "langchain==0.1.20" \
  "langchain-core==0.1.53" \
  "langchain-community==0.0.38" \
  "langchain-text-splitters==0.0.2"

# Repair NumPy/Pandas binary compatibility
!pip -q install --force-reinstall --no-cache-dir "numpy==1.26.4" "pandas==2.2.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 58.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.1/303.1 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 100.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.10 requires langchain-core>=1.0.0, but you have langchain-core 0.1.53 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
google-cloud-bigquery 3.41.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
langgraph-checkpoint 4.0.2 requires langch

In [22]:
import numpy as np
import pandas as pd
import torch
import faiss
import transformers

print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("FAISS imported")
print("Transformers imported")

NumPy: 1.26.4
Pandas: 2.2.2
Torch: 2.10.0+cu128
CUDA available: True
FAISS imported
Transformers imported


In [23]:
from ragatouille import RAGPretrainedModel
from langchain.retrievers.document_compressors.base import BaseDocumentCompressor

print("RAGatouille import successful")
print("LangChain retrievers module available")

/tmp/ipykernel_7464/2550023095.py:1: UserWarning: 
********************************************************************************
RAGatouille WARNING: Future Release Notice
--------------------------------------------
RAGatouille version 0.0.10 will be migrating to a PyLate backend 
instead of the current Stanford ColBERT backend.
PyLate is a fully mature, feature-equivalent backend, that greatly facilitates compatibility.
However, please pin version <0.0.10 if you require the Stanford ColBERT backend.
********************************************************************************
  from ragatouille import RAGPretrainedModel


RAGatouille import successful
LangChain retrievers module available


# Verify Cleaned Source Code

Before running the experiment, verify:

- DPR uses OriginalDPRRetriever
- MiniLM fallback is removed
- ColBERT patch exists
- Safe evaluator uses MakiGenerator

In [24]:
from retrievers.factory import get_retriever

print(type(get_retriever("bm25")))
print(type(get_retriever("dpr")))
print(type(get_retriever("contriever")))
print(type(get_retriever("colbertv2")))

<class 'retrievers.bm25_retriever.BM25Retriever'>
<class 'retrievers.dpr_original_retriever.OriginalDPRRetriever'>
<class 'retrievers.dense_retriever.ContrieverRetriever'>
<class 'retrievers.colbert_retriever.ColBERTRetriever'>


In [25]:
!grep -R "all-MiniLM-L6-v2" retrievers/ || true

In [26]:
!grep -n "safe_mark_tied_weights_as_initialized" retrievers/colbert_retriever.py

34:            def safe_mark_tied_weights_as_initialized(self):
39:            PreTrainedModel.mark_tied_weights_as_initialized = safe_mark_tied_weights_as_initialized


In [27]:
!grep -n "MakiGenerator" eval_all_retrievers_safe.py
!grep -n "from openai import OpenAI" eval_all_retrievers_safe.py || true

86:                from generator import MakiGenerator
88:                generator_obj = MakiGenerator(


# Set Fixed Mannheim Maki Generator

The generator must stay fixed for all retrievers.

We use:

- MAKI_HOST = https://maki.uni-mannheim.de/v1
- MAKI_MODEL = ministral-3-14b
- temperature = 0.0
- max_tokens = 512

Do not change these values during the experiment.

In [28]:
import os
from getpass import getpass

os.environ["MAKI_API_KEY"] = getpass("Paste Mannheim Maki API key: ")
os.environ["MAKI_HOST"] = "https://maki.uni-mannheim.de/v1"
os.environ["MAKI_MODEL"] = "ministral-3-14b"
os.environ["MAKI_DEFAULT_CTX"] = "7680"

print("Model:", os.environ["MAKI_MODEL"])
print("Host:", os.environ["MAKI_HOST"])
print("API key set:", bool(os.environ["MAKI_API_KEY"]))

Paste Mannheim Maki API key: ··········
Model: ministral-3-14b
Host: https://maki.uni-mannheim.de/v1
API key set: True


In [29]:
SMOKE_DIR = "/content/drive/MyDrive/medical_rag_sprint1_results/fresh_smoke_test"

!rm -rf "$SMOKE_DIR"
!mkdir -p "$SMOKE_DIR"

print("Smoke test output dir:", SMOKE_DIR)

Smoke test output dir: /content/drive/MyDrive/medical_rag_sprint1_results/fresh_smoke_test


# 5-Question Smoke Test

This will still index the full corpus for DPR, Contriever, and ColBERTv2.

Expected time:
- DPR indexing: around 1–2 minutes
- Contriever indexing: can take 20–30 minutes
- ColBERTv2 indexing: several minutes

This is normal.

Do not run the full 1000-question experiment until this test passes.

In [30]:
!python eval_all_retrievers_safe.py \
  --top_k 5 \
  --retrievers bm25 dpr contriever colbertv2 \
  --with-generation \
  --limit 5 \
  --results_dir "/content/drive/MyDrive/medical_rag_sprint1_results/fresh_smoke_test"

Loading data...
Data already prepared. Loading from disk...
Limited evaluation to first 5 questions.

Evaluating retriever: bm25
[bm25] Building/loading index...
Building BM25 index...
BM25 index saved.
[bm25] Indexing complete in 0.14 seconds.
Fixed generator initialized once.
Generator model: ministral-3-14b
Generator host: https://maki.uni-mannheim.de/v1
[bm25] Evaluating question 1/5...

Evaluating retriever: dpr
[dpr] Building/loading index...
Loading ORIGINAL DPR models (facebook dual-encoder)
  Question encoder : facebook/dpr-question_encoder-single-nq-base
  Context encoder  : facebook/dpr-ctx_encoder-single-nq-base
  Device           : cuda
Loading weights: 100% 197/197 [00:00<00:00, 915.61it/s, Materializing param=question_encoder.bert_model.encoder.layer.11.output.dense.weight]
DPRQuestionEncoder LOAD REPORT from: facebook/dpr-question_encoder-single-nq-base
Key                                             | Status     |  | 
------------------------------------------------+--

In [31]:
import pandas as pd

smoke_summary_path = "/content/drive/MyDrive/medical_rag_sprint1_results/fresh_smoke_test/fullrag_summary_top5.csv"
smoke_summary = pd.read_csv(smoke_summary_path)

smoke_summary

,retriever,num_questions,top_k,index_time_sec,avg_retrieval_time_sec,avg_recall_at_k,avg_mrr,status,error,avg_em,avg_f1,avg_rouge_l
0,bm25,5,5,0.137002,0.013265,0.666667,0.85,OK,NaN,0.0,0.227846,0.123058
1,dpr,5,5,47.219609,0.028275,0.466667,1.00,OK,NaN,0.0,0.250462,0.137384
2,contriever,5,5,1563.646750,0.115126,0.733333,1.00,OK,NaN,0.0,0.262990,0.132442
3,colbertv2,5,5,222.694162,1.160796,0.700000,1.00,OK,NaN,0.0,0.244314,0.139745


In [32]:
import numpy as np
import faiss
import os

emb_path = "data/embeddings/dpr_embeddings.npy"
idx_path = "data/indices/dpr_faiss.index"

print("DPR embedding exists:", os.path.exists(emb_path))
print("DPR FAISS index exists:", os.path.exists(idx_path))

emb = np.load(emb_path)
index = faiss.read_index(idx_path)

print("DPR embedding shape:", emb.shape)
print("DPR FAISS dimension:", index.d)
print("DPR index size:", index.ntotal)

DPR embedding exists: True
DPR FAISS index exists: True
DPR embedding shape: (3358, 768)
DPR FAISS dimension: 768
DPR index size: 3358


# Full Run

Only run this after the smoke test passes.

The evaluator is resumable:
- If Colab disconnects, rerun the same command.
- Completed rows are skipped.
- Results are saved directly to Google Drive.

In [33]:
FULL_DIR = "/content/drive/MyDrive/medical_rag_sprint1_results/fresh_full_run"

!mkdir -p "$FULL_DIR"

!python eval_all_retrievers_safe.py \
  --top_k 5 \
  --retrievers bm25 dpr contriever colbertv2 \
  --with-generation \
  --results_dir "$FULL_DIR"

Loading data...
Data already prepared. Loading from disk...
Evaluating on all 1000 questions.

Evaluating retriever: bm25
[bm25] Building/loading index...
Loading BM25 index from disk...
[bm25] Indexing complete in 0.05 seconds.
Fixed generator initialized once.
Generator model: ministral-3-14b
Generator host: https://maki.uni-mannheim.de/v1
[bm25] Evaluating question 1/1000...
[bm25] Evaluating question 10/1000...
[bm25] Evaluating question 20/1000...
[bm25] Evaluating question 30/1000...
[bm25] Evaluating question 40/1000...
[bm25] Evaluating question 50/1000...
[bm25] Evaluating question 60/1000...
[bm25] Evaluating question 70/1000...
[bm25] Evaluating question 80/1000...
[bm25] Evaluating question 90/1000...
[bm25] Evaluating question 100/1000...
[bm25] Evaluating question 110/1000...
[bm25] Evaluating question 120/1000...
[bm25] Evaluating question 130/1000...
[bm25] Evaluating question 140/1000...
[bm25] Evaluating question 150/1000...
[bm25] Evaluating question 160/1000...
[bm2

In [34]:
import pandas as pd

final_summary_path = "/content/drive/MyDrive/medical_rag_sprint1_results/fresh_full_run/fullrag_summary_top5.csv"
final_summary = pd.read_csv(final_summary_path)

final_summary.sort_values(["avg_recall_at_k", "avg_mrr"], ascending=False)

,retriever,num_questions,top_k,index_time_sec,avg_retrieval_time_sec,avg_recall_at_k,avg_mrr,status,error,avg_em,avg_f1,avg_rouge_l
3,colbertv2,1000,5,7.829190,0.040318,0.750586,0.979450,OK,NaN,0.0,0.233590,0.154276
2,contriever,1000,5,0.131166,0.120376,0.605600,0.952733,OK,NaN,0.0,0.230435,0.154575
0,bm25,1000,5,0.049768,0.015274,0.599211,0.911517,OK,NaN,0.0,0.224472,0.151236
1,dpr,1000,5,0.027159,0.019815,0.267880,0.474600,OK,NaN,0.0,0.168399,0.115600


In [35]:
!cd "/content/drive/MyDrive/medical_rag_sprint1_results" && \
zip -r fresh_full_run_results.zip fresh_full_run

  adding: fresh_full_run/ (stored 0%)
  adding: fresh_full_run/EXPERIMENT_METADATA.json (deflated 45%)
  adding: fresh_full_run/fullrag_bm25_top5.csv (deflated 68%)
  adding: fresh_full_run/fullrag_dpr_top5.csv (deflated 68%)
  adding: fresh_full_run/fullrag_contriever_top5.csv (deflated 69%)
  adding: fresh_full_run/fullrag_colbertv2_top5.csv (deflated 68%)
  adding: fresh_full_run/fullrag_summary_top5.csv (deflated 46%)
  adding: fresh_full_run/fullrag_ranked_retriever_comparison_top5.csv (deflated 46%)


In [36]:
from google.colab import files

files.download("/content/drive/MyDrive/medical_rag_sprint1_results/fresh_full_run_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
import os
import pandas as pd
import numpy as np

BASE_DIR = "/content/drive/MyDrive/medical_rag_sprint1_results/fresh_full_run"
OUT_DIR = "/content/drive/MyDrive/medical_rag_sprint1_results/fresh_final_csv_outputs"

os.makedirs(OUT_DIR, exist_ok=True)

FIXED_GENERATOR_PROVIDER = "Mannheim Maki OpenAI-compatible API"
FIXED_LLM_MODEL = "ministral-3-14b"
FIXED_TEMPERATURE = 0.0
FIXED_MAX_TOKENS = 512
FIXED_TOP_K = 5
FIXED_DATASET = "PubMedQA-derived project corpus"
FIXED_NUM_QUESTIONS = 1000
FIXED_PROMPT_TEMPLATE = "Same project RAG prompt template used for all retrievers"

detail_files = {
    "bm25": os.path.join(BASE_DIR, "fullrag_bm25_top5.csv"),
    "dpr": os.path.join(BASE_DIR, "fullrag_dpr_top5.csv"),
    "contriever": os.path.join(BASE_DIR, "fullrag_contriever_top5.csv"),
    "colbertv2": os.path.join(BASE_DIR, "fullrag_colbertv2_top5.csv"),
}

summary_path = os.path.join(BASE_DIR, "fullrag_summary_top5.csv")

required = list(detail_files.values()) + [summary_path]
missing = [p for p in required if not os.path.exists(p)]

if missing:
    raise FileNotFoundError("Missing files:\n" + "\n".join(missing))

summary = pd.read_csv(summary_path)
summary["retriever"] = summary["retriever"].astype(str).str.lower()

summary_ranked = summary.sort_values(
    ["avg_recall_at_k", "avg_mrr"],
    ascending=False
).reset_index(drop=True)

summary_ranked["retrieval_rank"] = summary_ranked.index + 1
best_retriever = summary_ranked.loc[0, "retriever"]

TIME_ESTIMATES = {
    "bm25": {"estimated_avg_retrieval_time_sec": 0.014},
    "dpr": {"estimated_avg_retrieval_time_sec": 0.016},
    "contriever": {"estimated_avg_retrieval_time_sec": 0.119},
    "colbertv2": {"estimated_avg_retrieval_time_sec": 0.035},
}

# 1. Individual results
frames = []

for retriever, path in detail_files.items():
    df = pd.read_csv(path)
    df["retriever"] = retriever
    df["retriever_display_name"] = df["retriever"].replace({
        "bm25": "BM25",
        "dpr": "Original DPR",
        "contriever": "Contriever",
        "colbertv2": "ColBERTv2",
    })

    df["fixed_generator_provider"] = FIXED_GENERATOR_PROVIDER
    df["fixed_llm_model"] = FIXED_LLM_MODEL
    df["fixed_temperature"] = FIXED_TEMPERATURE
    df["fixed_max_tokens"] = FIXED_MAX_TOKENS
    df["fixed_top_k"] = FIXED_TOP_K
    df["fixed_dataset"] = FIXED_DATASET
    df["fixed_prompt_template"] = FIXED_PROMPT_TEMPLATE
    df["is_best_retriever"] = df["retriever"] == best_retriever
    df["best_retriever_for_experiment"] = best_retriever
    df["estimated_avg_retrieval_time_sec"] = TIME_ESTIMATES[retriever]["estimated_avg_retrieval_time_sec"]
    df["estimated_total_retrieval_time_sec_for_1000_questions"] = (
        df["estimated_avg_retrieval_time_sec"] * FIXED_NUM_QUESTIONS
    )
    df["generation_time_note"] = "LLM generation time was not separately logged per row."

    frames.append(df)

individual = pd.concat(frames, ignore_index=True)
individual.to_csv(os.path.join(OUT_DIR, "01_individual_results_top5.csv"), index=False)

# 2. Comparison results
comparison = summary_ranked.copy()
comparison["retriever_display_name"] = comparison["retriever"].replace({
    "bm25": "BM25",
    "dpr": "Original DPR",
    "contriever": "Contriever",
    "colbertv2": "ColBERTv2",
})
comparison["fixed_generator_provider"] = FIXED_GENERATOR_PROVIDER
comparison["fixed_llm_model"] = FIXED_LLM_MODEL
comparison["fixed_temperature"] = FIXED_TEMPERATURE
comparison["fixed_max_tokens"] = FIXED_MAX_TOKENS
comparison["fixed_top_k"] = FIXED_TOP_K
comparison["fixed_dataset"] = FIXED_DATASET
comparison["is_best_retriever"] = comparison["retriever"] == best_retriever
comparison["best_retriever_for_experiment"] = best_retriever
comparison["selection_reason"] = np.where(
    comparison["is_best_retriever"],
    "Selected as best retriever because it achieved the highest Recall@5 and MRR under the fixed-generator setup.",
    "Not selected because another retriever achieved stronger retrieval performance."
)
comparison["evaluation_warning"] = (
    "Exact Match is unsuitable for long-form generated answers. "
    "F1 and ROUGE-L are weak lexical indicators. "
    "Future work should add faithfulness, hallucination, coverage, and diversity metrics."
)
comparison.to_csv(os.path.join(OUT_DIR, "02_comparison_results_top5.csv"), index=False)

# 3. Best suggestion
best = comparison[comparison["retriever"] == best_retriever].iloc[0]

best_suggestion = pd.DataFrame([{
    "best_retriever": best["retriever"],
    "best_retriever_display_name": best["retriever_display_name"],
    "avg_recall_at_k": best["avg_recall_at_k"],
    "avg_mrr": best["avg_mrr"],
    "avg_em": best["avg_em"],
    "avg_f1": best["avg_f1"],
    "avg_rouge_l": best["avg_rouge_l"],
    "fixed_generator_provider": FIXED_GENERATOR_PROVIDER,
    "fixed_llm_model": FIXED_LLM_MODEL,
    "recommendation": (
        f"Use {best['retriever_display_name']} as the strongest Sprint 1 baseline retriever because it achieved "
        f"the highest Recall@5 ({best['avg_recall_at_k']:.6f}) and MRR ({best['avg_mrr']:.6f})."
    ),
    "important_limitation": (
        "Exact Match is not useful for long-form generated answers. "
        "Future work should include faithfulness, hallucination, coverage, and diversity metrics."
    ),
}])

best_suggestion.to_csv(os.path.join(OUT_DIR, "03_best_retriever_suggestion_top5.csv"), index=False)

print("Saved final CSV outputs to:", OUT_DIR)
!ls "$OUT_DIR"

Saved final CSV outputs to: /content/drive/MyDrive/medical_rag_sprint1_results/fresh_final_csv_outputs
01_individual_results_top5.csv	03_best_retriever_suggestion_top5.csv
02_comparison_results_top5.csv


In [38]:
!cd "/content/drive/MyDrive/medical_rag_sprint1_results" && \
zip -r fresh_final_csv_outputs.zip fresh_final_csv_outputs

from google.colab import files
files.download("/content/drive/MyDrive/medical_rag_sprint1_results/fresh_final_csv_outputs.zip")

  adding: fresh_final_csv_outputs/ (stored 0%)
  adding: fresh_final_csv_outputs/01_individual_results_top5.csv (deflated 73%)
  adding: fresh_final_csv_outputs/02_comparison_results_top5.csv (deflated 66%)
  adding: fresh_final_csv_outputs/03_best_retriever_suggestion_top5.csv (deflated 33%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>